# Simulation figuresRegenerates the four figures used in the manuscript: loss trajectories, and train/test C-index and AMSE across epochs.## About this notebookThe model, loss, sampler, metrics and data generation all live in the`rnn_agt` package. This notebook sets up an experiment and reports it; nothingis redefined here, so every notebook and driver in the repository shares oneimplementation.

In [ ]:
import os, syssys.path.insert(0, os.path.abspath(".."))   # repository root, so `rnn_agt` importsimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport rnn_agtfrom rnn_agt import data as Dfrom rnn_agt.seeds import make_seedsfrom rnn_agt.train import TrainConfig, train_model, predictfrom rnn_agt.metrics import evaluateprint("rnn_agt", rnn_agt.__version__)

In [ ]:
ERROR_DISTS = ["normal", "gumbel", "logistic"]MEAN_FUNCS  = ["linear", "interaction", "gam"]def run_with_history(mean_func, error_dist, censoring=0.50, n_train=1000,                     n_test=2000, epochs=15, seed=42):    seeds = make_seeds(seed)    rng = seeds.data()    tau = D.calibrate_tau(n_train, D.MEAN_FUNCTIONS[mean_func], error_dist,                          rng, censoring, D.DEPENDENCE_SPECS["ar1"])    tr = D.make_dataset(n_train, mean_func, error_dist, rng,                        dependence="ar1", tau=tau)    te = D.make_dataset(n_test, mean_func, error_dist, rng,                        dependence="ar1", tau=tau)    cfg = TrainConfig(model="rnn_agt", epochs=epochs, pair_sample_s=30,                      hidden_dim=64, gru_layers=2, lr=3e-4, track_history=True)    return train_model(tr, te, 3, cfg, make_seeds(11))histories = {}for mf in MEAN_FUNCS:    for ed in ERROR_DISTS:        histories[(mf, ed)] = run_with_history(mf, ed)        print(f"{mf:12s} {ed:9s} done", flush=True)

### Figure 1: loss trajectories

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)for ax, mf in zip(axes, MEAN_FUNCS):    for ed in ERROR_DISTS:        ax.plot(range(1, len(histories[(mf, ed)].epoch_losses) + 1),                histories[(mf, ed)].epoch_losses, marker="o", ms=3, label=ed)    ax.set_title(mf); ax.set_xlabel("epoch"); ax.grid(alpha=.3)axes[0].set_ylabel("Gehan-WRS training loss")axes[0].legend(title="error")fig.suptitle("Training loss across epochs")fig.tight_layout(); fig.savefig("loss_trajectorie.png", dpi=150); plt.show()

### Figures 2 and 3: train vs test across epochs

In [ ]:
for metric, fname, ylabel in (    ("cindex", "train_test_cindex.png", "IPCW C-index"),    ("amse",   "train_test_amse.png",   "AMSE"),):    fig, axes = plt.subplots(1, 3, figsize=(14, 4))    for ax, mf in zip(axes, MEAN_FUNCS):        h = histories[(mf, "normal")].history        epochs = range(1, len(h[f"train_{metric}"]) + 1)        ax.plot(epochs, h[f"train_{metric}"], marker="o", ms=3, label="train")        ax.plot(epochs, h[f"test_{metric}"],  marker="s", ms=3, label="test")        ax.set_title(mf); ax.set_xlabel("epoch"); ax.grid(alpha=.3)    axes[0].set_ylabel(ylabel); axes[0].legend()    fig.suptitle(f"Train vs test {ylabel} (Normal errors, AR(1), 50% censoring)")    fig.tight_layout(); fig.savefig(fname, dpi=150); plt.show()print("Where the test curve sits above the train curve, that is the effect "      "described in Section 5.2: G is re-estimated per partition, so the IPCW "      "weights are on different scales and the two are not the same quantity.")

### Figure 4: AMSE against C-index

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))markers = {"linear": "o", "interaction": "s", "gam": "^"}colors  = {"normal": "tab:blue", "gumbel": "tab:orange", "logistic": "tab:green"}for (mf, ed), res in histories.items():    ax.scatter(res.metrics["test_cindex"], res.metrics["test_amse"],               marker=markers[mf], c=colors[ed], s=70,               label=f"{mf}, {ed}")ax.set_xlabel("test IPCW C-index"); ax.set_ylabel("test AMSE")ax.grid(alpha=.3)ax.legend(fontsize=7, ncol=2)ax.set_title("Discrimination against calibration")fig.tight_layout(); fig.savefig("amse_c-index_plots.png", dpi=150); plt.show()